[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/07_ONNX_Runtime/01_ORT_Architecture/ORT_Architecture_Deep_Dive.ipynb)

# ORT Architecture — Deep Dive

## Table of Contents
| # | Section | Description |
|---|---------|-------------|
| 1 | [Overview](#1) | Historical context and design philosophy |
| 2 | [Layered Architecture](#2) | API → Graph Partitioner → EP Framework → Kernels |
| 3 | [API Layer](#3) | Language bindings, SessionOptions, RunOptions |
| 4 | [Graph Partitioner](#4) | Node assignment, cost model, fallback chains |
| 5 | [Execution Provider Framework](#5) | EP interface, capability declaration, kernel registration |
| 6 | [Kernel Layer](#6) | Operator implementations, dispatch tables |
| 7 | [Memory Arena](#7) | Arena allocators, buffer planning, lifetime analysis |
| 8 | [Threading Model](#8) | Intra-op vs inter-op parallelism |
| 9 | [Session Lifecycle](#9) | Creation, optimization, execution pipeline |
| 10 | [Visualization](#10) | Architecture diagrams and throughput analysis |

In [ ]:
# Install dependencies
!pip install onnxruntime onnx numpy matplotlib -q

<a id='1'></a>
## 1. Overview and Design Philosophy

**ONNX Runtime (ORT)** is Microsoft's cross-platform, high-performance inference engine for ONNX models. Unlike framework-specific runtimes (e.g., TorchScript, TF-Lite), ORT was designed from the ground up as a **graph-level scheduler** that delegates operator execution to pluggable hardware backends called **Execution Providers (EPs)**.

The fundamental throughput equation governing ORT's design:

$$\text{Throughput} = \frac{\text{batch\_size}}{\text{latency}(\text{batch\_size})} \quad \text{[samples/sec]}$$

Where latency itself decomposes into:

$$\text{latency} = T_{\text{kernel}} + T_{\text{memory}} + T_{\text{dispatch}} + T_{\text{sync}}$$

ORT minimizes each component through:
- **$T_{\text{kernel}}$**: Fused operator kernels that reduce compute redundancy
- **$T_{\text{memory}}$**: Arena allocators and buffer reuse that eliminate malloc overhead
- **$T_{\text{dispatch}}$**: Graph partitioning that minimizes EP boundary crossings
- **$T_{\text{sync}}$**: Asynchronous execution and IOBinding for zero-copy I/O

### Historical Context

ORT emerged from Microsoft's internal needs to serve models across Azure services at scale. The key insight was that framework-specific inference (PyTorch eager, TF SavedModel) left performance on the table because they couldn't perform cross-operator optimizations at the graph level. By standardizing on ONNX as the IR and building a dedicated inference engine, Microsoft achieved:

1. **Framework agnosticism** — Train in PyTorch, JAX, or TensorFlow; deploy uniformly
2. **Hardware abstraction** — Same model binary runs on CPU, GPU, NPU, FPGA via EPs
3. **Production hardening** — Stable C ABI, deterministic memory, thread-safe sessions

<a id='2'></a>
## 2. Layered Architecture

ORT is organized into four distinct layers, each with clear responsibilities and interfaces:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         APPLICATION LAYER                                │
│   Python │ C# │ Java │ JavaScript │ C/C++ │ Rust │ Swift │ Objective-C   │
├─────────────────────────────────────────────────────────────────────────┤
│                          API LAYER                                       │
│   InferenceSession │ SessionOptions │ RunOptions │ IOBinding             │
│   ModelMetadata │ OrtValue │ MemoryInfo │ Allocator                      │
├─────────────────────────────────────────────────────────────────────────┤
│                     GRAPH PARTITIONER LAYER                              │
│   GraphTransformer Pipeline │ Node Assignment │ Subgraph Fusion          │
│   Cost Model │ EP Capability Query │ Fallback Resolution                 │
├─────────────────────────────────────────────────────────────────────────┤
│                 EXECUTION PROVIDER FRAMEWORK                             │
│   ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐    │
│   │   CPU    │ │   CUDA   │ │ TensorRT │ │ OpenVINO │ │ DirectML │    │
│   │  EP      │ │   EP     │ │   EP     │ │   EP     │ │   EP     │    │
│   └──────────┘ └──────────┘ └──────────┘ └──────────┘ └──────────┘    │
├─────────────────────────────────────────────────────────────────────────┤
│                        KERNEL LAYER                                      │
│   MLAS (CPU GEMM/Conv) │ cuDNN │ cuBLAS │ oneDNN │ Vendor Libraries      │
│   Custom Op Kernels │ Contrib Ops │ Fused Kernels                        │
└─────────────────────────────────────────────────────────────────────────┘
```

### Data Flow Through Layers

When you call `session.run()`, the following pipeline executes:

```
User Code                 API Layer              Graph Partitioner        EP Framework         Kernels
   │                         │                        │                      │                   │
   │──── run(feeds) ────────►│                        │                      │                   │
   │                         │── validate inputs ────►│                      │                   │
   │                         │                        │── fetch exec plan ──►│                   │
   │                         │                        │                      │── dispatch op ───►│
   │                         │                        │                      │                   │── execute
   │                         │                        │                      │◄── result ────────│
   │                         │◄────── outputs ────────│◄─────────────────────│                   │
   │◄── OrtValue[] ─────────│                        │                      │                   │
```

The key insight is that **graph optimization and partitioning happen once** (at session creation), while **execution follows a pre-computed plan** on each `run()` call. This amortizes the O(n) graph analysis cost over potentially millions of inference calls.

<a id='3'></a>
## 3. API Layer

The API layer provides the user-facing contract. All language bindings ultimately call the same C API (`onnxruntime_c_api.h`), ensuring consistent behavior across platforms.

### Core Abstractions

| Abstraction | Purpose | Lifecycle |
|-------------|---------|----------|
| `InferenceSession` | Holds optimized graph + execution plan | Long-lived (app lifetime) |
| `SessionOptions` | Configures optimization level, threading, EPs | Set once before session creation |
| `RunOptions` | Per-invocation flags (logging, termination) | Created per `run()` call |
| `OrtValue` | Tensor container with memory ownership info | Per-tensor, may alias user memory |
| `IOBinding` | Pre-bound I/O for zero-copy GPU inference | Reused across runs |
| `MemoryInfo` | Describes where memory lives (device, allocator) | Static descriptors |

### SessionOptions Parameters

The `SessionOptions` object controls the optimization-performance tradeoff:

$$\text{session\_creation\_time} \propto \text{optimization\_level} \times |\text{nodes}|$$

Higher optimization levels perform more graph transformations (constant folding, fusion, layout optimization), increasing session creation time but reducing per-inference latency. The total cost over $N$ inferences:

$$\text{Total Cost} = T_{\text{create}} + N \cdot T_{\text{run}}(\text{opt\_level})$$

For $N \gg 1$ (production serving), aggressive optimization is always justified:

$$\lim_{N \to \infty} \frac{T_{\text{create}}}{N} = 0$$

In [ ]:
import onnxruntime as ort
import numpy as np

print(f"ONNX Runtime version: {ort.__version__}")
print(f"Available Execution Providers: {ort.get_available_providers()}")
print(f"\nDevice info:")
print(f"  Available devices: {ort.get_device()}")

so = ort.SessionOptions()
print(f"\nDefault SessionOptions:")
print(f"  graph_optimization_level: {so.graph_optimization_level}")
print(f"  intra_op_num_threads: {so.intra_op_num_threads}")
print(f"  inter_op_num_threads: {so.inter_op_num_threads}")
print(f"  execution_mode: {so.execution_mode}")
print(f"  enable_mem_pattern: {so.enable_mem_pattern}")
print(f"  enable_cpu_mem_arena: {so.enable_cpu_mem_arena}")
print(f"  enable_profiling: {so.enable_profiling}")

<a id='4'></a>
## 4. Graph Partitioner — The Core Scheduler

The graph partitioner is ORT's most architecturally significant component. It solves a constrained assignment problem: given an ordered list of Execution Providers $[EP_0, EP_1, \ldots, EP_k]$ (where $EP_k$ is always CPU), assign each node $n_i$ in the graph to exactly one EP such that:

1. **Capability constraint**: $EP_j$ claims $n_i$ only if it implements the required operator with matching dtypes and shape constraints
2. **Priority ordering**: $EP_0$ gets first refusal, then $EP_1$, etc.
3. **Minimized transfers**: Adjacent nodes on the same EP share memory without cross-device copies

### Formal Assignment Algorithm

```
for each node n_i in topological order:
    for j = 0 to k:
        if EP_j.CanExecute(n_i, dtype, shape):
            assign(n_i, EP_j)
            break
```

### Partitioning Creates Subgraphs

After assignment, contiguous runs of nodes assigned to the same EP form **subgraphs** (also called "fused nodes" or "compiled kernels" for EPs like TensorRT):

```
Original Graph:  [Conv]──[BN]──[Relu]──[Pool]──[Reshape]──[MatMul]──[Softmax]
                   │       │      │       │        │          │         │
EP Assignment:    GPU     GPU    GPU     GPU      CPU        GPU       GPU
                   └───────────────┘      │        │          └─────────┘
Subgraphs:        GPU_Subgraph_0       GPU_Sub_1  CPU_Node   GPU_Subgraph_2
```

### Cross-EP Transfer Cost

Each EP boundary incurs a data transfer cost:

$$T_{\text{transfer}} = T_{\text{sync}} + \frac{\text{tensor\_bytes}}{\text{bandwidth}_{\text{PCIe/NVLink}}}$$

For a tensor of shape $(B, C, H, W)$ in float32:

$$\text{bytes} = B \cdot C \cdot H \cdot W \cdot 4$$

With PCIe Gen4 x16 bandwidth of ~25 GB/s, a ResNet feature map of $(32, 256, 56, 56)$ requires:

$$T_{\text{transfer}} = \frac{32 \times 256 \times 56 \times 56 \times 4}{25 \times 10^9} \approx 4.1 \text{ ms}$$

This is why **minimizing EP boundary crossings** is critical for performance.

In [ ]:
import onnx
from onnx import helper, TensorProto, numpy_helper
import numpy as np

# Build a model to demonstrate partitioning concepts
# Simple: Input -> MatMul -> Add -> Relu -> MatMul -> Softmax
W1 = np.random.randn(784, 256).astype(np.float32)
B1 = np.random.randn(256).astype(np.float32)
W2 = np.random.randn(256, 10).astype(np.float32)
B2 = np.random.randn(10).astype(np.float32)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 10])

nodes = [
    helper.make_node("MatMul", ["X", "W1"], ["H1"]),
    helper.make_node("Add", ["H1", "B1"], ["H1b"]),
    helper.make_node("Relu", ["H1b"], ["H1r"]),
    helper.make_node("MatMul", ["H1r", "W2"], ["H2"]),
    helper.make_node("Add", ["H2", "B2"], ["H2b"]),
    helper.make_node("Softmax", ["H2b"], ["Y"], axis=1),
]

graph = helper.make_graph(
    nodes, "MLP", [X], [Y],
    initializer=[
        numpy_helper.from_array(W1, "W1"),
        numpy_helper.from_array(B1, "B1"),
        numpy_helper.from_array(W2, "W2"),
        numpy_helper.from_array(B2, "B2"),
    ]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
onnx.checker.check_model(model)
onnx.save(model, "mlp_demo.onnx")

print(f"Model created with {len(model.graph.node)} nodes")
for i, n in enumerate(model.graph.node):
    print(f"  Node {i}: {n.op_type} ({list(n.input)} -> {list(n.output)})")

In [ ]:
import onnxruntime as ort
import time

# Demonstrate session creation with different optimization levels
levels = {
    "DISABLED": ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
    "BASIC": ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
    "EXTENDED": ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,
    "ALL": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
}

creation_times = {}
for name, level in levels.items():
    so = ort.SessionOptions()
    so.graph_optimization_level = level
    
    start = time.perf_counter()
    sess = ort.InferenceSession("mlp_demo.onnx", so, providers=["CPUExecutionProvider"])
    creation_times[name] = (time.perf_counter() - start) * 1000
    
print("Session creation times by optimization level:")
for name, t in creation_times.items():
    print(f"  {name:10s}: {t:.3f} ms")

<a id='5'></a>
## 5. Execution Provider Framework

The EP framework is ORT's hardware abstraction layer. Each EP implements a well-defined interface that answers two questions:

1. **What can I execute?** — Via `GetCapability()` which returns supported node groups
2. **How do I execute it?** — Via registered kernel implementations

### EP Interface Contract

```
┌─────────────────────────────────────────────────────────┐
│              IExecutionProvider Interface                 │
├─────────────────────────────────────────────────────────┤
│  GetCapability(graph) → List[IndexedSubGraph]           │
│  Compile(subgraph) → CompiledKernel                     │
│  GetAllocator(mem_type) → Allocator                     │
│  GetKernelRegistry() → KernelRegistry                   │
│  GetDataTransfer() → DataTransfer                       │
│  Type() → "CUDAExecutionProvider" | ...                 │
└─────────────────────────────────────────────────────────┘
```

### Capability Declaration

When an EP claims a set of nodes, it returns an `IndexedSubGraph` — a contiguous region of the ONNX graph that the EP can execute as a unit. For compilation-based EPs (TensorRT, OpenVINO), this entire subgraph gets compiled into an opaque engine:

$$\text{SubGraph} \xrightarrow{\text{Compile()}} \text{OpaqueEngine}$$

The compiled engine replaces the original nodes with a single "fused node" whose kernel simply invokes the pre-compiled engine. This amortizes compilation cost over all future `run()` calls.

### EP Priority and Fallback

EPs are registered in priority order. The fallback chain ensures graceful degradation:

```
TensorRT EP ──(unsupported ops)──► CUDA EP ──(no GPU)──► CPU EP
     │                                  │                    │
     ▼                                  ▼                    ▼
  Compiled TRT                     cuDNN/cuBLAS           MLAS/Eigen
  engines for                      kernels for            kernels for
  supported ops                    remaining ops          all ops
```

A single model may have nodes scattered across multiple EPs. ORT automatically inserts **memory copy nodes** (MemcpyFromHost, MemcpyToHost) at EP boundaries to handle data transfer.

In [ ]:
import onnxruntime as ort

# Inspect available EPs and their properties
providers = ort.get_available_providers()
print("Available Execution Providers:")
print("=" * 50)
for i, ep in enumerate(providers):
    print(f"  Priority {i}: {ep}")

# Create session and inspect which EP was used for each node
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
sess = ort.InferenceSession("mlp_demo.onnx", so, providers=["CPUExecutionProvider"])

print(f"\nSession providers: {sess.get_providers()}")
print(f"\nModel inputs:")
for inp in sess.get_inputs():
    print(f"  {inp.name}: {inp.type} shape={inp.shape}")
print(f"\nModel outputs:")
for out in sess.get_outputs():
    print(f"  {out.name}: {out.type} shape={out.shape}")

<a id='6'></a>
## 6. Kernel Layer

At the bottom of the stack, kernels are the actual operator implementations. Each EP maintains a **kernel registry** — a dispatch table mapping `(op_type, op_version, dtypes)` to concrete implementations.

### CPU Kernel Stack (MLAS)

For CPU inference, ORT uses **MLAS** (Microsoft Linear Algebra Subroutines), a hand-tuned library providing:

- **GEMM**: Tiled, cache-blocked matrix multiplication with AVX-512/NEON intrinsics
- **Conv**: im2col + GEMM or direct Winograd convolutions
- **Pooling/Activations**: Vectorized elementwise operations

The performance of a GEMM kernel for matrices $A \in \mathbb{R}^{M \times K}$ and $B \in \mathbb{R}^{K \times N}$:

$$\text{FLOPS} = 2MKN \quad \text{(multiply-accumulate)}$$

$$\text{Arithmetic Intensity} = \frac{2MKN}{4(MK + KN + MN)} \quad \text{[FLOP/byte]}$$

When arithmetic intensity exceeds the machine's ops:byte ratio (the "ridge point" on the roofline model), the kernel becomes compute-bound rather than memory-bound.

### Kernel Dispatch Table

```
┌───────────────────────────────────────────────────────────┐
│              CPU Kernel Registry (partial)                 │
├──────────────┬──────────────┬────────────────────────────┤
│  Op Type     │  Versions    │  Supported Types            │
├──────────────┼──────────────┼────────────────────────────┤
│  MatMul      │  1-13        │  float, float16, double    │
│  Conv        │  1-11        │  float, float16            │
│  Relu        │  6-14        │  float, float16, int8      │
│  Softmax     │  1-13        │  float, double             │
│  Attention   │  contrib     │  float, float16 (fused)    │
└──────────────┴──────────────┴────────────────────────────┘
```

### Fused Kernels

ORT's graph optimizer can recognize multi-node patterns and replace them with fused kernels. A fused `MatMul+Add+Relu` kernel avoids writing the intermediate result to memory:

$$\text{Unfused memory traffic} = 2MN + 2MN + MN = 5MN \text{ reads+writes}$$
$$\text{Fused memory traffic} = MK + KN + MN \text{ (input + weight + output only)}$$

<a id='7'></a>
## 7. Memory Arena — Buffer Planning and Lifetime Analysis

ORT's memory subsystem is one of its key performance differentiators. Rather than calling `malloc`/`free` for each intermediate tensor, ORT uses **arena allocators** with pre-planned buffer layouts.

### Arena Allocation Strategy

```
┌──────────────── Memory Arena (pre-allocated contiguous block) ────────────────┐
│                                                                                │
│  ┌─────────┐  ┌───────────────────┐  ┌──────┐  ┌────────────────┐           │
│  │Tensor A │  │    Tensor B       │  │ T_C  │  │   Tensor D     │  (free)   │
│  │ (alive) │  │   (alive)         │  │(dead)│  │   (alive)      │           │
│  └─────────┘  └───────────────────┘  └──────┘  └────────────────┘           │
│                                                                                │
│  After T_C is consumed, its slot is available for future tensors              │
└────────────────────────────────────────────────────────────────────────────────┘
```

### Tensor Lifetime Analysis

ORT performs a static analysis pass to determine tensor lifetimes — the interval $[t_{\text{produce}}, t_{\text{last\_consume}}]$ for each intermediate value. Two tensors with non-overlapping lifetimes can share the same memory slot:

$$\text{can\_share}(T_a, T_b) \iff [t_a^{\text{start}}, t_a^{\text{end}}] \cap [t_b^{\text{start}}, t_b^{\text{end}}] = \emptyset$$

This is equivalent to the **interval graph coloring** problem:

$$\text{min\_buffers} = \max_{t} |\{T : t \in [T.\text{start}, T.\text{end}]\}|$$

The minimum number of buffers needed equals the maximum number of simultaneously live tensors.

### Memory Pattern Optimization

When `enable_mem_pattern = True` (default), ORT records the allocation pattern during the first `run()` and reuses it for subsequent calls. This eliminates allocation decisions entirely from the hot path:

$$T_{\text{alloc}}^{\text{first\_run}} = O(n) \quad \text{(plan creation)}$$
$$T_{\text{alloc}}^{\text{subsequent}} = O(1) \quad \text{(plan replay)}$$

### Worked Example

Consider a 3-node graph: $A \to B \to C \to D$ where each operation produces a tensor of 1MB:

| Time Step | Live Tensors | Peak Memory |
|-----------|-------------|-------------|
| $t_0$ | Input (1MB) | 1 MB |
| $t_1$ | Input + A_out (2MB) | 2 MB |
| $t_2$ | A_out + B_out (2MB) | 2 MB |
| $t_3$ | B_out + C_out (2MB) | 2 MB |
| $t_4$ | Output (1MB) | 1 MB |

Peak memory = 2 MB with buffer reuse, vs 5 MB without (naive allocation).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize tensor lifetime and memory reuse
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Tensor lifetimes in a sample graph
tensors = {
    'Input': (0, 1),
    'Conv_out': (1, 2),
    'BN_out': (2, 3),
    'Relu_out': (3, 5),
    'Pool_out': (4, 6),
    'FC_out': (5, 7),
    'Softmax_out': (6, 8),
}

colors = plt.cm.Set3(np.linspace(0, 1, len(tensors)))
for i, (name, (start, end)) in enumerate(tensors.items()):
    ax1.barh(i, end - start, left=start, height=0.6, color=colors[i], edgecolor='black', linewidth=0.5)
    ax1.text(start + (end-start)/2, i, name, ha='center', va='center', fontsize=8, fontweight='bold')

ax1.set_xlabel('Execution Time Step', fontsize=11)
ax1.set_ylabel('Tensor', fontsize=11)
ax1.set_title('Tensor Lifetimes in ORT Execution Plan', fontsize=12, fontweight='bold')
ax1.set_yticks(range(len(tensors)))
ax1.set_yticklabels([''] * len(tensors))
ax1.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Peak liveness')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='x')

# Right: Memory comparison (arena vs naive)
nodes = ['4 nodes', '8 nodes', '16 nodes', '32 nodes', '64 nodes']
naive_mem = [4, 8, 16, 32, 64]  # MB (no reuse)
arena_mem = [2, 3, 5, 8, 12]    # MB (with lifetime-based reuse)

x = np.arange(len(nodes))
width = 0.35

ax2.bar(x - width/2, naive_mem, width, label='Naive (no reuse)', color='#ff6b6b', edgecolor='black', linewidth=0.5)
ax2.bar(x + width/2, arena_mem, width, label='Arena (lifetime reuse)', color='#4ecdc4', edgecolor='black', linewidth=0.5)

ax2.set_xlabel('Graph Size', fontsize=11)
ax2.set_ylabel('Peak Memory (MB)', fontsize=11)
ax2.set_title('Memory Savings with Arena Allocation', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(nodes)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

for i in range(len(nodes)):
    savings = (1 - arena_mem[i]/naive_mem[i]) * 100
    ax2.annotate(f'{savings:.0f}%\nsaved', xy=(x[i] + width/2, arena_mem[i]),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=8, color='green')

plt.tight_layout()
plt.savefig('ort_memory_arena.png', dpi=150, bbox_inches='tight')
plt.show()
print("Memory arena visualization saved.")

<a id='8'></a>
## 8. Threading Model — Intra-op vs Inter-op Parallelism

ORT provides two orthogonal threading knobs that control different parallelism dimensions:

### Intra-op Threading

Parallelizes work **within** a single operator. For a GEMM computing $C = A \times B$ where $C \in \mathbb{R}^{M \times N}$, the work is divided across $T$ threads:

$$\text{Work per thread} = \frac{2MKN}{T}$$

$$\text{Ideal speedup} = \min\left(T, \frac{2MKN}{\text{cache\_line\_ops}}\right)$$

In practice, Amdahl's Law limits the benefit due to sequential overhead:

$$S(T) = \frac{1}{(1-p) + \frac{p}{T}}$$

where $p$ is the parallelizable fraction of the operator.

### Inter-op Threading

Parallelizes execution of **independent operators** in the graph. For a DAG with width $w$ (maximum number of operators executable in parallel):

$$\text{Critical path time} = \sum_{i \in \text{longest\_path}} T_i$$

$$\text{Speedup}_{\text{inter}} \leq \frac{\sum_{i=1}^{n} T_i}{\text{Critical path time}}$$

### When to Use Which

```
Sequential Chain (typical CNN):       Wide DAG (multi-branch):

  [Conv] → [BN] → [Relu]             [Branch_A]  [Branch_B]  [Branch_C]
     │                                     │           │           │
     └─ intra-op helps here                └───────────┴───────────┘
        (parallelize the Conv)                         │
                                              inter-op helps here
                                              (run branches in parallel)
```

**Rule of thumb**: For transformer/CNN inference on CPU, start with `intra_op_num_threads = physical_cores` and `inter_op_num_threads = 1`. Only increase inter-op for highly parallel architectures (e.g., Inception-style models).

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# Benchmark different threading configurations
thread_configs = [
    (1, 1), (2, 1), (4, 1), (8, 1),  # Vary intra-op
    (4, 2), (4, 4),  # Vary inter-op with fixed intra
]

results = []
test_input = np.random.randn(32, 784).astype(np.float32)

for intra, inter in thread_configs:
    so = ort.SessionOptions()
    so.intra_op_num_threads = intra
    so.inter_op_num_threads = inter
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    sess = ort.InferenceSession("mlp_demo.onnx", so, providers=["CPUExecutionProvider"])
    
    # Warmup
    for _ in range(10):
        sess.run(None, {"X": test_input})
    
    # Benchmark
    latencies = []
    for _ in range(100):
        start = time.perf_counter()
        sess.run(None, {"X": test_input})
        latencies.append((time.perf_counter() - start) * 1000)
    
    results.append({
        'intra': intra, 'inter': inter,
        'mean_ms': np.mean(latencies),
        'p50_ms': np.percentile(latencies, 50),
        'p99_ms': np.percentile(latencies, 99),
    })

print(f"{'Intra':>5} {'Inter':>5} {'Mean(ms)':>10} {'P50(ms)':>10} {'P99(ms)':>10}")
print("-" * 45)
for r in results:
    print(f"{r['intra']:>5} {r['inter']:>5} {r['mean_ms']:>10.3f} {r['p50_ms']:>10.3f} {r['p99_ms']:>10.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize threading impact
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Intra-op scaling
intra_results = [r for r in results if r['inter'] == 1]
threads = [r['intra'] for r in intra_results]
latencies = [r['mean_ms'] for r in intra_results]
ideal_scaling = [latencies[0] / t for t in threads]

ax1.plot(threads, latencies, 'bo-', linewidth=2, markersize=8, label='Actual latency')
ax1.plot(threads, ideal_scaling, 'r--', linewidth=1.5, alpha=0.7, label='Ideal linear scaling')
ax1.set_xlabel('Intra-op Threads', fontsize=11)
ax1.set_ylabel('Mean Latency (ms)', fontsize=11)
ax1.set_title('Intra-op Thread Scaling\n(inter_op=1, batch=32)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(threads)

# Right: Amdahl's Law illustration
T = np.arange(1, 33)
for p in [0.5, 0.75, 0.9, 0.95, 0.99]:
    speedup = 1 / ((1 - p) + p / T)
    ax2.plot(T, speedup, linewidth=2, label=f'p = {p}')

ax2.set_xlabel('Number of Threads', fontsize=11)
ax2.set_ylabel('Speedup', fontsize=11)
ax2.set_title("Amdahl's Law: $S(T) = \\frac{1}{(1-p) + p/T}$", fontsize=12, fontweight='bold')
ax2.legend(fontsize=10, title='Parallel fraction')
ax2.grid(True, alpha=0.3)
ax2.set_xlim([1, 32])

plt.tight_layout()
plt.savefig('ort_threading.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='9'></a>
## 9. Session Lifecycle — From Model to Execution

The complete lifecycle of an ORT session involves distinct phases, each with its own computational cost and opportunities for optimization:

```
Phase 1: MODEL LOADING                    Phase 2: GRAPH OPTIMIZATION
┌──────────────────────────┐              ┌──────────────────────────┐
│  Read .onnx protobuf     │              │  Level 1: Basic          │
│  Deserialize ModelProto   │              │    - Constant folding    │
│  Validate IR structure    │──────────────►    - Dead code removal   │
│  Build internal IR        │              │    - Redundant node elim │
│  Register initializers    │              │  Level 2: Extended       │
└──────────────────────────┘              │    - Op fusion            │
                                          │    - Layout optimization  │
                                          │  Level 3: Layout          │
                                          │    - NHWC transforms      │
                                          └────────────┬─────────────┘
                                                       │
Phase 3: PARTITIONING                     Phase 4: EXECUTION PLANNING
┌──────────────────────────┐              ┌──────────────────────────┐
│  Query each EP            │              │  Allocate memory arenas  │
│  Assign nodes to EPs      │◄─────────────  Compute tensor lifetimes │
│  Insert copy nodes        │              │  Build execution order   │
│  Compile EP subgraphs     │              │  Record memory pattern   │
└──────────────────────────┘              └──────────────────────────┘
                                                       │
                                                       ▼
                                          Phase 5: STEADY-STATE EXECUTION
                                          ┌──────────────────────────┐
                                          │  Bind inputs             │
                                          │  Execute plan (kernels)  │
                                          │  Collect outputs         │
                                          │  (Repeat for each run)   │
                                          └──────────────────────────┘
```

### Cost Profile (Typical)

| Phase | Time | Frequency |
|-------|------|----------|
| Loading | 10-500 ms | Once |
| Optimization | 50-2000 ms | Once |
| Partitioning | 10-5000 ms (compilation-based EPs) | Once |
| Planning | 1-10 ms | Once |
| Execution | 0.1-100 ms | Per request |

The key engineering insight: **invest heavily in phases 1-4** (which run once) to minimize phase 5 (which runs millions of times).

In [ ]:
import onnxruntime as ort
import numpy as np
import time

# Demonstrate session lifecycle timing
print("=" * 60)
print("SESSION LIFECYCLE TIMING")
print("=" * 60)

# Phase 1+2+3+4: Session creation
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.enable_profiling = True

start = time.perf_counter()
sess = ort.InferenceSession("mlp_demo.onnx", so, providers=["CPUExecutionProvider"])
creation_time = (time.perf_counter() - start) * 1000
print(f"\nSession creation (load + optimize + partition + plan): {creation_time:.3f} ms")

# Phase 5: Execution (first run - includes memory pattern recording)
x = np.random.randn(1, 784).astype(np.float32)
start = time.perf_counter()
_ = sess.run(None, {"X": x})
first_run = (time.perf_counter() - start) * 1000
print(f"First run (includes pattern recording): {first_run:.3f} ms")

# Subsequent runs (steady state)
latencies = []
for _ in range(1000):
    start = time.perf_counter()
    _ = sess.run(None, {"X": x})
    latencies.append((time.perf_counter() - start) * 1000)

print(f"\nSteady-state execution (1000 runs):")
print(f"  Mean: {np.mean(latencies):.4f} ms")
print(f"  P50:  {np.percentile(latencies, 50):.4f} ms")
print(f"  P95:  {np.percentile(latencies, 95):.4f} ms")
print(f"  P99:  {np.percentile(latencies, 99):.4f} ms")
print(f"  Std:  {np.std(latencies):.4f} ms")

throughput = 1000 / np.mean(latencies)  # samples/sec for batch=1
print(f"\nThroughput (batch=1): {throughput:.0f} samples/sec")

# Disable profiling
prof_file = sess.end_profiling()
print(f"\nProfile saved to: {prof_file}")

<a id='10'></a>
## 10. Architecture Visualization

Let's create comprehensive visualizations of the ORT architecture and performance characteristics.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('ONNX Runtime — Layered Architecture', fontsize=16, fontweight='bold', pad=20)

# Layer colors
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

# Application Layer
rect = FancyBboxPatch((0.5, 8.5), 13, 1.2, boxstyle="round,pad=0.1", 
                       facecolor=colors[0], alpha=0.3, edgecolor=colors[0], linewidth=2)
ax.add_patch(rect)
ax.text(7, 9.3, 'APPLICATION LAYER', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(7, 8.8, 'Python  |  C#  |  Java  |  JavaScript  |  C/C++  |  Rust', 
        ha='center', va='center', fontsize=9)

# API Layer
rect = FancyBboxPatch((0.5, 6.8), 13, 1.4, boxstyle="round,pad=0.1",
                       facecolor=colors[1], alpha=0.3, edgecolor=colors[1], linewidth=2)
ax.add_patch(rect)
ax.text(7, 7.7, 'API LAYER', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(7, 7.2, 'InferenceSession | SessionOptions | RunOptions | IOBinding | OrtValue', 
        ha='center', va='center', fontsize=9)

# Graph Partitioner Layer
rect = FancyBboxPatch((0.5, 4.8), 13, 1.7, boxstyle="round,pad=0.1",
                       facecolor=colors[2], alpha=0.3, edgecolor=colors[2], linewidth=2)
ax.add_patch(rect)
ax.text(7, 6.0, 'GRAPH PARTITIONER', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(7, 5.5, 'Graph Transformers | Node Assignment | Cost Model | Subgraph Fusion', 
        ha='center', va='center', fontsize=9)
ax.text(7, 5.1, 'Memory Planning | Tensor Lifetime Analysis | Execution Order', 
        ha='center', va='center', fontsize=9)

# EP Framework Layer
rect = FancyBboxPatch((0.5, 2.8), 13, 1.7, boxstyle="round,pad=0.1",
                       facecolor=colors[3], alpha=0.3, edgecolor=colors[3], linewidth=2)
ax.add_patch(rect)
ax.text(7, 4.0, 'EXECUTION PROVIDER FRAMEWORK', ha='center', va='center', fontsize=12, fontweight='bold')

# EP boxes
ep_names = ['CPU', 'CUDA', 'TensorRT', 'OpenVINO', 'DirectML', 'CoreML']
ep_colors = ['#ecf0f1', '#76d7c4', '#f9e79f', '#aed6f1', '#d7bde2', '#fadbd8']
for i, (name, color) in enumerate(zip(ep_names, ep_colors)):
    x_pos = 1.0 + i * 2.1
    rect = FancyBboxPatch((x_pos, 3.0), 1.8, 0.7, boxstyle="round,pad=0.05",
                           facecolor=color, edgecolor='gray', linewidth=1)
    ax.add_patch(rect)
    ax.text(x_pos + 0.9, 3.35, name, ha='center', va='center', fontsize=8, fontweight='bold')

# Kernel Layer
rect = FancyBboxPatch((0.5, 0.5), 13, 2.0, boxstyle="round,pad=0.1",
                       facecolor=colors[4], alpha=0.3, edgecolor=colors[4], linewidth=2)
ax.add_patch(rect)
ax.text(7, 2.0, 'KERNEL LAYER', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(7, 1.4, 'MLAS (CPU GEMM/Conv) | cuDNN | cuBLAS | oneDNN | Vendor Libs', 
        ha='center', va='center', fontsize=9)
ax.text(7, 0.9, 'Fused Kernels | Custom Ops | Contrib Ops | Quantized Kernels', 
        ha='center', va='center', fontsize=9)

# Arrows between layers
for y in [8.5, 6.8, 4.8, 2.8]:
    ax.annotate('', xy=(7, y), xytext=(7, y + 0.15),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

plt.tight_layout()
plt.savefig('ort_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print("Architecture diagram saved.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Throughput vs Batch Size analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]

# Simulate realistic latency behavior
# Latency grows sub-linearly with batch size (due to parallelism)
base_latency = 0.5  # ms for batch=1
latencies = [base_latency * (1 + 0.3 * np.log2(max(b, 1))) for b in batch_sizes]
throughputs = [b / (l / 1000) for b, l in zip(batch_sizes, latencies)]

# Plot 1: Latency vs Batch Size
axes[0].plot(batch_sizes, latencies, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Batch Size', fontsize=11)
axes[0].set_ylabel('Latency (ms)', fontsize=11)
axes[0].set_title('Latency vs Batch Size', fontsize=12, fontweight='bold')
axes[0].set_xscale('log', base=2)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(batch_sizes)
axes[0].set_xticklabels(batch_sizes)

# Plot 2: Throughput vs Batch Size
axes[1].plot(batch_sizes, throughputs, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Batch Size', fontsize=11)
axes[1].set_ylabel('Throughput (samples/sec)', fontsize=11)
axes[1].set_title('Throughput = batch_size / latency', fontsize=12, fontweight='bold')
axes[1].set_xscale('log', base=2)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(batch_sizes)
axes[1].set_xticklabels(batch_sizes)

# Plot 3: Latency percentile distribution
np.random.seed(42)
latency_samples = np.random.lognormal(mean=np.log(0.5), sigma=0.3, size=1000)
percentiles = [50, 75, 90, 95, 99, 99.9]
percentile_values = [np.percentile(latency_samples, p) for p in percentiles]

axes[2].bar(range(len(percentiles)), percentile_values, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Percentile', fontsize=11)
axes[2].set_ylabel('Latency (ms)', fontsize=11)
axes[2].set_title('Latency Percentile Distribution', fontsize=12, fontweight='bold')
axes[2].set_xticks(range(len(percentiles)))
axes[2].set_xticklabels([f'P{int(p)}' for p in percentiles])
axes[2].grid(True, alpha=0.3, axis='y')

for i, v in enumerate(percentile_values):
    axes[2].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('ort_throughput_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Throughput analysis saved.")

## 11. Key Equations Summary

### Performance Metrics

$$\text{Throughput} = \frac{\text{batch\_size}}{\text{latency}} \quad [\text{samples/sec}]$$

$$\text{Latency}_{\text{P99}} = \text{quantile}(\text{latencies}, 0.99)$$

### Memory Efficiency

$$\text{Arena utilization} = \frac{\text{peak\_live\_bytes}}{\text{arena\_size}} \leq 1$$

$$\text{Buffer savings} = 1 - \frac{\text{arena\_peak}}{\sum_i \text{size}(T_i)}$$

### Threading Efficiency

$$S(T) = \frac{1}{(1-p) + \frac{p}{T}} \quad \text{(Amdahl's Law)}$$

$$\text{Efficiency}(T) = \frac{S(T)}{T} = \frac{1}{T(1-p) + p}$$

### Transfer Overhead

$$T_{\text{H2D}} = T_{\text{launch}} + \frac{N \cdot \text{sizeof}(\text{dtype})}{\text{BW}_{\text{PCIe}}}$$

In [ ]:
# Cleanup
import os
for f in ['mlp_demo.onnx']:
    if os.path.exists(f):
        os.remove(f)

# Remove any profiling files
for f in os.listdir('.'):
    if f.endswith('.json') and 'onnxruntime_profile' in f:
        os.remove(f)

print("Cleanup complete.")

## Summary

ONNX Runtime's architecture is organized into four layers:

1. **API Layer** — User-facing abstractions (`InferenceSession`, `SessionOptions`, `IOBinding`) that provide language-agnostic access through a stable C ABI

2. **Graph Partitioner** — Assigns nodes to Execution Providers using priority-based capability queries, minimizing cross-device transfers

3. **Execution Provider Framework** — Hardware abstraction layer where each EP declares capabilities, compiles subgraphs, and manages device-specific memory

4. **Kernel Layer** — Concrete operator implementations (MLAS, cuDNN, oneDNN) including fused kernels that reduce memory traffic

The **Memory Arena** system uses tensor lifetime analysis to achieve near-optimal buffer reuse, while the **threading model** provides both intra-op (within-operator) and inter-op (across-operator) parallelism knobs for tuning to specific hardware topologies.